# 04b – Genetic Algorithm + ANN Hybrid (Option A: ANN Hyperparameter Optimization)
**Owner:** Member 4 (Deep Learning and Hybrid Engineer)

Implements **Option A**: A Genetic Algorithm searches over ANN architectural and training hyperparameters (`n_layers`, `hidden_units`, `lr`, `dropout`, `batch_size`, `weight_decay`) using **Validation RMSE** as the fitness criterion (lower RMSE = better fitness). Test data is never used during evolution.

In [1]:
import copy
import json
import random
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, MODELS_DIR, RANDOM_SEED, RESULTS_DIR
from src.evaluate import (
    compute_metrics,
    plot_parity,
    plot_residuals,
    save_model_run_metadata,
    save_predictions,
)
from src.features import build_preprocessor, get_xy

def set_seeds(seed: int = RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seeds(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

X_train_df, y_train_s, id_train = get_xy("train")
X_val_df, y_val_s, id_val = get_xy("val")
X_test_df, y_test_s, id_test = get_xy("test")

preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(X_train_df).astype(np.float32)
X_val = preprocessor.transform(X_val_df).astype(np.float32)
X_test = preprocessor.transform(X_test_df).astype(np.float32)
y_train = y_train_s.to_numpy(dtype=np.float32)
y_val = y_val_s.to_numpy(dtype=np.float32)
y_test = y_test_s.to_numpy(dtype=np.float32)

In [2]:
class DynamicANN(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU()])
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def decode_individual(ind: dict) -> list[int]:
    """Convert chromosome (n_layers, first_width) into decaying hidden layer sizes."""
    dims = []
    w = ind["width"]
    for _ in range(ind["n_layers"]):
        dims.append(max(16, int(w)))
        w = w // 2
    return dims


def evaluate_ann_candidate(ind: dict, max_epochs: int = 120, patience: int = 15, seed: int = RANDOM_SEED):
    set_seeds(seed)
    hidden_dims = decode_individual(ind)
    model = DynamicANN(X_train.shape[1], hidden_dims, ind["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=ind["lr"], weight_decay=ind["weight_decay"])
    criterion = nn.MSELoss()

    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=ind["batch_size"],
        shuffle=True,
        generator=g,
    )
    X_val_t = torch.from_numpy(X_val).to(DEVICE)
    y_val_t = torch.from_numpy(y_val).to(DEVICE)

    best_val_mse = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    no_imp = 0
    t0 = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_mse = float(criterion(model(X_val_t), y_val_t).item())
        if val_mse < best_val_mse - 1e-5:
            best_val_mse = val_mse
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    train_time = time.perf_counter() - t0
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t).cpu().numpy()
    val_rmse = compute_metrics(y_val, val_pred)["rmse"]
    return val_rmse, model, val_pred, train_time, best_epoch

In [3]:
# Required GA modular functions (Section 10)
SEARCH_SPACE = {
    "n_layers": [2, 3, 4],
    "width": [64, 128, 192, 256],
    "lr": [3e-4, 5e-4, 1e-3, 2e-3],
    "dropout": [0.0, 0.1, 0.2, 0.3],
    "batch_size": [32, 64, 128],
    "weight_decay": [1e-5, 1e-4, 5e-4, 1e-3],
}

_FITNESS_CACHE: dict[tuple, float] = {}

def create_individual() -> dict:
    return {k: random.choice(v) for k, v in SEARCH_SPACE.items()}

def create_population(pop_size: int) -> list[dict]:
    return [create_individual() for _ in range(pop_size)]

def fitness(individual: dict) -> float:
    """Fitness = Validation RMSE (lower is better)."""
    key = tuple(sorted(individual.items()))
    if key not in _FITNESS_CACHE:
        val_rmse, _, _, _, _ = evaluate_ann_candidate(individual)
        _FITNESS_CACHE[key] = float(val_rmse)
    return _FITNESS_CACHE[key]

def selection(population: list[dict], fitness_scores: list[float], tournament_k: int = 3) -> dict:
    """Tournament selection (minimizing RMSE)."""
    candidates = random.sample(list(zip(population, fitness_scores)), k=min(tournament_k, len(population)))
    winner = min(candidates, key=lambda pair: pair[1])[0]
    return copy.deepcopy(winner)

def crossover(parent1: dict, parent2: dict, crossover_rate: float = 0.8) -> tuple[dict, dict]:
    """Uniform gene crossover."""
    if random.random() > crossover_rate:
        return copy.deepcopy(parent1), copy.deepcopy(parent2)
    child1, child2 = {}, {}
    for gene in SEARCH_SPACE:
        if random.random() < 0.5:
            child1[gene], child2[gene] = parent1[gene], parent2[gene]
        else:
            child1[gene], child2[gene] = parent2[gene], parent1[gene]
    return child1, child2

def mutation(individual: dict, mutation_rate: float = 0.25) -> dict:
    """Mutate genes independently within SEARCH_SPACE."""
    mutated = copy.deepcopy(individual)
    for gene, choices in SEARCH_SPACE.items():
        if random.random() < mutation_rate:
            mutated[gene] = random.choice(choices)
    return mutated

def elitism(population: list[dict], fitness_scores: list[float], n_elites: int = 2) -> list[dict]:
    """Preserve the top n_elites individuals with lowest RMSE."""
    ranked = sorted(zip(population, fitness_scores), key=lambda pair: pair[1])
    return [copy.deepcopy(ind) for ind, _ in ranked[:n_elites]]

def run_genetic_algorithm(pop_size: int = 8, n_generations: int = 6, n_elites: int = 2):
    set_seeds(RANDOM_SEED)
    population = create_population(pop_size)
    history = []
    best_overall = None
    best_overall_fit = float("inf")

    for gen in range(1, n_generations + 1):
        scores = [fitness(ind) for ind in population]
        gen_best = float(min(scores))
        gen_mean = float(np.mean(scores))
        gen_best_ind = population[int(np.argmin(scores))]
        if gen_best < best_overall_fit:
            best_overall_fit = gen_best
            best_overall = copy.deepcopy(gen_best_ind)

        history.append({
            "generation": gen,
            "best_fitness_val_rmse": round(gen_best, 5),
            "mean_fitness_val_rmse": round(gen_mean, 5),
            "best_so_far_val_rmse": round(best_overall_fit, 5),
        })
        print(f"Gen {gen:2d}/{n_generations} | Best Val RMSE: {gen_best:.4f} | Mean: {gen_mean:.4f} | Best Ind: {gen_best_ind}")

        if gen == n_generations:
            break

        next_pop = elitism(population, scores, n_elites=n_elites)
        while len(next_pop) < pop_size:
            p1 = selection(population, scores)
            p2 = selection(population, scores)
            c1, c2 = crossover(p1, p2)
            next_pop.append(mutation(c1))
            if len(next_pop) < pop_size:
                next_pop.append(mutation(c2))
        population = next_pop

    return best_overall, best_overall_fit, pd.DataFrame(history)

In [4]:
t_ga_start = time.perf_counter()
best_ind, best_ga_rmse, history_df = run_genetic_algorithm(pop_size=8, n_generations=6, n_elites=2)
ga_search_time = time.perf_counter() - t_ga_start

history_df.to_csv(RESULTS_DIR / "ga_history.csv", index=False)
(RESULTS_DIR / "ga_best_ann_config.json").write_text(
    json.dumps({"chromosome": best_ind, "hidden_dims": decode_individual(best_ind), "best_val_rmse": best_ga_rmse}, indent=2),
    encoding="utf-8",
)

# Convergence plot
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(history_df["generation"], history_df["best_fitness_val_rmse"], "o-", label="Generation Best Val RMSE", linewidth=2)
ax.plot(history_df["generation"], history_df["mean_fitness_val_rmse"], "s--", label="Population Mean Val RMSE", linewidth=1.8)
ax.set_title("Genetic Algorithm Convergence (Option A: ANN Hyperparameter Optimization)")
ax.set_xlabel("Generation")
ax.set_ylabel("Fitness: Validation RMSE (lower is better)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "ga_convergence.png", dpi=200)
plt.close(fig)

# Final training & evaluation of the GA-optimized ANN
val_rmse, ga_model, val_pred, final_fit_time, best_epoch = evaluate_ann_candidate(best_ind, max_epochs=180, patience=20)
ga_model.eval()
with torch.no_grad():
    test_pred = ga_model(torch.from_numpy(X_test).to(DEVICE)).cpu().numpy()

print(f"\nGA Search Time: {ga_search_time:.2f} s | Final Fit Time: {final_fit_time:.2f} s | Best Epoch: {best_epoch}")
print("GA-ANN Hidden Dims:", decode_individual(best_ind))
print("GA-ANN Val Metrics :", compute_metrics(y_val, val_pred))
print("GA-ANN Test Metrics:", compute_metrics(y_test, test_pred))

torch.save({"state_dict": ga_model.state_dict(), "config": best_ind, "hidden_dims": decode_individual(best_ind)}, MODELS_DIR / "ga_ann_best.pt")
save_predictions(id_val, y_val, val_pred, "ga_ann", "val")
save_predictions(id_test, y_test, test_pred, "ga_ann", "test")
save_model_run_metadata(
    model_name="ga_ann",
    display_name="GA-ANN Hybrid",
    training_time_sec=final_fit_time,
    search_time_sec=ga_search_time,
    extra={"best_chromosome": best_ind, "hidden_dims": decode_individual(best_ind), "best_epoch": best_epoch},
)
plot_parity(y_test, test_pred, "ga_ann", split="test")
plot_residuals(y_test, test_pred, "ga_ann", split="test")

Gen  1/6 | Best Val RMSE: 1.0973 | Mean: 1.1262 | Best Ind: {'n_layers': 2, 'width': 192, 'lr': 0.0003, 'dropout': 0.0, 'batch_size': 64, 'weight_decay': 1e-05}


Gen  2/6 | Best Val RMSE: 1.0973 | Mean: 1.1268 | Best Ind: {'n_layers': 2, 'width': 192, 'lr': 0.0003, 'dropout': 0.0, 'batch_size': 64, 'weight_decay': 1e-05}


Gen  3/6 | Best Val RMSE: 1.0938 | Mean: 1.1153 | Best Ind: {'n_layers': 2, 'width': 192, 'lr': 0.0003, 'dropout': 0.3, 'batch_size': 64, 'weight_decay': 0.0005}
Gen  4/6 | Best Val RMSE: 1.0938 | Mean: 1.1146 | Best Ind: {'n_layers': 2, 'width': 192, 'lr': 0.0003, 'dropout': 0.3, 'batch_size': 64, 'weight_decay': 0.0005}


Gen  5/6 | Best Val RMSE: 1.0938 | Mean: 1.1082 | Best Ind: {'n_layers': 2, 'width': 192, 'lr': 0.0003, 'dropout': 0.3, 'batch_size': 64, 'weight_decay': 0.0005}


Gen  6/6 | Best Val RMSE: 1.0922 | Mean: 1.1293 | Best Ind: {'n_layers': 2, 'width': 192, 'lr': 0.0003, 'dropout': 0.3, 'batch_size': 64, 'weight_decay': 0.0001}



GA Search Time: 173.18 s | Final Fit Time: 12.13 s | Best Epoch: 89
GA-ANN Hidden Dims: [192, 96]
GA-ANN Val Metrics : {'r2': 0.5766109572954241, 'rmse': 1.0921966759489958, 'mae': 0.7545072951388235}
GA-ANN Test Metrics: {'r2': 0.5530829925225964, 'rmse': 0.9781109835177156, 'mae': 0.6878979696488629}


WindowsPath('C:/Users/nourh/.gemini/antigravity/scratch/mof-co2-uptake-prediction/figures/residuals_ga_ann_test.png')